# Sorting Algorithms Analysis

This notebook compares execution time, comparison counts, and swap/move counts for multiple sorting algorithms across random, sorted, and reversed inputs.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df_time = pd.read_csv("data/results_all_scenarios.csv")
df_ops = pd.read_csv("data/results_operations.csv")
df_time["time_ms"] = df_time["time_s"] * 1000
df_time = df_time.dropna(subset=["time_ms"])
df_random = df_time[df_time["scenario"] == "random"].copy()
nlogn_algos = ["Merge Sort", "Quick Sort", "Heap Sort", "Shell Sort", "Tim Sort"]

## Execution Time by Scenario

In [ ]:
fig = px.line(
    df_time,
    x="size",
    y="time_ms",
    color="algorithm",
    facet_col="scenario",
    markers=True,
    title="Sorting Algorithms — Time vs Input Size by Scenario",
    labels={"size": "Input Size (n)", "time_ms": "Time (ms)", "algorithm": "Algorithm"},
    template="plotly_dark",
)
fig.update_xaxes(tickformat=",")
fig.show()

## O(n log n) Algorithms on Random Input

In [ ]:
df_nlogn = df_random[df_random["algorithm"].isin(nlogn_algos)]
fig = px.line(
    df_nlogn,
    x="size",
    y="time_ms",
    color="algorithm",
    markers=True,
    title="O(n log n) Algorithms — Random Input",
    labels={"size": "Input Size (n)", "time_ms": "Time (ms)"},
    template="plotly_dark",
)
fig.update_xaxes(tickformat=",")
fig.show()

## Best Case vs Worst Case Sensitivity

In [ ]:
pivot = df_time.pivot_table(index=["algorithm", "size"], columns="scenario", values="time_ms").reset_index()
pivot["delta_ms"] = pivot["reversed"] - pivot["sorted"]
pivot = pivot.dropna(subset=["delta_ms"])
fig = px.line(
    pivot,
    x="size",
    y="delta_ms",
    color="algorithm",
    markers=True,
    title="Sensitivity to Input Order: reversed_time - sorted_time",
    labels={"size": "Input Size (n)", "delta_ms": "Delta Time (ms)"},
    template="plotly_dark",
)
fig.update_xaxes(tickformat=",")
fig.show()

## Actual Operation Counts

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Comparisons", "Swaps / Moves"])
for algorithm_name in df_ops["algorithm"].unique():
    subset = df_ops[df_ops["algorithm"] == algorithm_name]
    fig.add_trace(go.Scatter(x=subset["size"], y=subset["comparisons"], mode="lines+markers", name=algorithm_name), row=1, col=1)
    fig.add_trace(go.Scatter(x=subset["size"], y=subset["swaps"], mode="lines+markers", name=algorithm_name, showlegend=False), row=1, col=2)
fig.update_layout(title="Instrumented Operation Counts vs Input Size", template="plotly_dark")
fig.update_xaxes(tickformat=",")
fig.show()

## Log-Log View

In [ ]:
df_log = df_random.copy()
df_log["log_n"] = np.log2(df_log["size"])
df_log["log_t"] = np.log2(df_log["time_ms"].clip(lower=0.001))
fig = px.line(
    df_log,
    x="log_n",
    y="log_t",
    color="algorithm",
    markers=True,
    title="Log-Log Plot for Random Input",
    labels={"log_n": "log2(n)", "log_t": "log2(time ms)"},
    template="plotly_dark",
)
x_range = [df_log["log_n"].min(), df_log["log_n"].max()]
for slope, label, color in [(1.0, "O(n)", "lime"), (1.5, "O(n log n)", "cyan"), (2.0, "O(n^2)", "red")]:
    center_x = sum(x_range) / 2
    center_y = df_log["log_t"].mean()
    intercept = center_y - slope * center_x
    fig.add_trace(go.Scatter(x=x_range, y=[slope * x + intercept for x in x_range], mode="lines", line=dict(dash="dash", color=color), name=label))
fig.show()